In [115]:
import os
from typing import List, Dict
from bs4 import BeautifulSoup
from datetime import datetime
import pandas as pd
from pathlib import Path

def str_getfrom_token_first(s1: str, tok: str):
    if not tok: return str
    pos = s1.find(tok)
    return s1[pos:] if pos >= 0 else s1

CLEANUP_CHARS = "､––—...!!;:()[]" 
TRANS_TABLE = str.maketrans('', '', CLEANUP_CHARS)

class BibleHubHebrewHTMLParser:
    def __init__(self):
        pass
    
    def parse_html_folder_biblehub(self, folder_path: str, book_filter: List[str]= None, save_result=False) -> Dict[str, List[List[str]]]:
        """Parse HTML files from BibleHub"""
        result = {}
        sentence_tokens = []
        df_result = []
        iterator_folders = []
        if book_filter is None:
            iterator_folders = os.listdir(folder_path)
        else:
            #if any(book.lower() in folder.lower() for book in book_filter)
            #dircont =
            iterator_folders = [n.lower() for n in os.listdir(folder_path) if os.path.isdir(os.path.join(folder_path, n))
                       and n.lower() in book_filter]
            #for book in book_filter:
            #    if book.lower()  in dircont: iterator_folders.append(book)
        urls_greek = {}
        chdone = 0
        totchs = len(iterator_folders)
        for fold in iterator_folders:
            chapsnames = [filename for filename in os.listdir(folder_path.strip(os.path.sep)+os.path.sep+fold) 
                          if filename.endswith('.htm') or filename.endswith('.html')]
            ccchdone = 0
            totchhh = len(chapsnames)
            for filename in chapsnames:
                if(True):
                    chapter = os.path.splitext(filename)[0]
                    #print(chapter)
                    #if chapter != "3":
                    #    return result
                    current_book = fold
                    result[f"{current_book}_{chapter}"] = []
                    print(f"{current_book} {chapter} ( {float(chdone)/float(totchs)*float(100)+(float(1)/float(totchs))*(float(ccchdone)/float(totchhh))*float(100):.1f}% )")
                    soup = BeautifulSoup(open(os.path.join(folder_path, fold, filename)), 'html.parser')
                    
                    table_spans = soup.find_all('table', class_='tablefloatheb')
                    curr_verse = 1
                    word_idx = 0
                    for tbc in range(len(table_spans)):
                        #reset
                        strong_num=-1
                        strong_title = ''
                        strong_en_title = ''
                        translit=''
                        translit_title=''
                        form = ''
                        form_en = ''
                        curr_node = None
                        #start parsing
                        table = table_spans[tbc]
                        #verse number?
                        curr_node = table.find("span", class_='refheb')
                        if curr_node is not None:
                            # nbsp;
                            tx = curr_node.text.strip("\xa0")
                            if tx.isnumeric():
                                curr_verse = int(tx)
                                word_idx = 0
                            else:
                                print(tx)
                                c_node = table.find("span", class_='hebrew')
                                if c_node is not None:
                                    print(f"no REFTOP:: {current_book}_{chapter}__{curr_verse} :: {c_node.text.replace("\xa0", ' ')}")
                        
                        if(len(result[f"{current_book}_{chapter}"])<curr_verse):
                            for jj in range(len(result[f"{current_book}_{chapter}"]),curr_verse):
                                result[f"{current_book}_{chapter}"].append([])
                        
                        #hebrew text
                        hebrew_nodes = table.find_all("span", class_='hebrew')
                        collected_parts = []
                        
                        for node in hebrew_nodes:
                            # 'separator' parametrs nodrošina, ka vārdi nesalīp kopā, ja starp tiem nav atstarpes
                            txt = node.get_text(" ", strip=True)
                            if txt:
                                collected_parts.append(txt)
                        
                        # Saliekam kopā. Ebreju tekstam reizēm vajag reversed() atkarībā no tā, kā tabula uzbūvēta
                        form = " ".join(collected_parts).replace("\xa0", " ").translate(TRANS_TABLE).strip()
                        #print(f"{not form} :: {form}")
                        if not form:
                            print(f"whitespaced/interpunct only:: {current_book}_{chapter}__{curr_verse}")
                            continue
                        
                        #en text
                        eng_nodes = table.find_all("span", class_='eng')
                        #curr_node = table.find("span", class_='eng')
                        collected_parts = []
                        for node in eng_nodes:
                            txt = node.get_text(" ", strip=True)
                            if txt:
                                collected_parts.append(txt)
                        form_en = " ".join(collected_parts).replace("\xa0", " ").strip()
                        
                        #strongs title?
                        str_nodes = table.find_all("span", class_='strongs')
                        collected_parts = []
                        collected_parts_titles = []
                        for n in str_nodes:
                            curr_node = n.find('a')
                            if curr_node is not None:
                                if curr_node.text.isnumeric():
                                    strong_num = int(curr_node.text)
                                    #collected_parts.append(strong_num)
                                collected_parts_titles.append(curr_node.text + str_getfrom_token_first(curr_node.attrs.get('title', ''), ':'))
                                collected_parts.append(curr_node.text)
                                attrb = curr_node.attrs.get('href', '')
                                if(attrb != ''):
                                    if(attrb.startswith('/hebrew')):
                                        urls_greek[attrb]=f"curl -O https://www.biblehub.com{attrb}"
                        strong_title = " ".join(collected_parts_titles).strip()
                        #if strong_num <1:
                        #    print(f"no strongs:: {current_book}_{chapter}__{curr_verse} :: {form}")
                        #only one strong number for now....
                        #strong_num = ", ".join(collected_parts).strip()
                                        
                        #englishmans concordance nooo its just same number as per strongs!!! therefore findlast
                        curr_node = table.find_all("span", class_='strongsnt')[-1]
                        if curr_node is not None:
                            curr_node = curr_node.find('a')
                            if curr_node is not None:
                                strong_en_title = curr_node.text.replace("\xa0", "")
                                attrb = curr_node.attrs.get('href', '')
                                if(attrb != ''):
                                    if(attrb.startswith('/hebrew')):
                                        urls_greek[attrb]=f"curl -O https://www.biblehub.com{attrb}"
                            
                        #translit
                        translit_nodes = table.find_all("span", class_='translit')
                        collected_parts = []
                        collected_parts_titles = []
                        for n in translit_nodes:
                            curr_node = n.find('a')
                            if curr_node is not None:
                                attrb = curr_node.attrs.get('href', '')
                                if(attrb != ''):
                                    if(attrb.startswith('/hebrew')):
                                        urls_greek[attrb]=f"curl -O https://www.biblehub.com{attrb}"
                                collected_parts_titles.append(curr_node.attrs.get('title', '').replace("\xa0", ' '))
                                collected_parts.append(curr_node.text.replace("\xa0", ' '))
                        translit_title = " ".join(collected_parts_titles).strip()
                        translit = " ".join(collected_parts).strip()

                                    
                        xdata = {
                        ######## text is inside form attribute(!)                        'text': token.text.strip(),
                        'verse': curr_verse,
                        'word': word_idx,
                        #'id' : token.attrib.get('id', ''),
                        'form' : form,
                        'form_en': form_en,
                        #'citation-part' : token.attrib.get('citation-part', ''),
                        #'lemma' : token.attrib.get('lemma', ''),
                        #'part-of-speech' : token.attrib.get('part-of-speech', ''),
                        #'morphology' : token.attrib.get('morphology', ''),
                        #'head-id' : token.attrib.get('head-id', ''),
                        #'relation' : token.attrib.get('relation', ''),
                        #'presentation-after' : token.attrib.get('presentation-after', ''),
                        #'information-status' : token.attrib.get('information-status', ''),
                        #'empty-token-sort' : token.attrib.get('empty-token-sort', ''),
                        #'antecedent-id' : token.attrib.get('antecedent-id', ''),
                        #'presentation-before' : token.attrib.get('presentation-before', ''),
                        #'contrast-group' : token.attrib.get('contrast-group', ''),
                        ####"attrs": dict(token.attrib)
                        'strong_num': strong_num,
                        'strong_title' : strong_title,
                        'strong_en_title' : strong_en_title,
                        'translit': translit,
                        'translit_title': translit_title
                    }
                        result[f"{current_book}_{chapter}"][curr_verse-1].append(xdata)
                        xdata['book'] = current_book
                        xdata['chapter'] = chapter
                        df_result.append(xdata)
                        word_idx += 1
                
                ccchdone +=1
                
            chdone += 1

        if(save_result):
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            filename = f"{timestamp}_hb_dload.sh"
            if(len(urls_greek.items())>0):
                with open(filename, "w") as file:
                    for k, v in urls_greek.items():
                        file.write(f"{v}\n")
            dbname = f"{timestamp}_biblehub_hb_en_direct.csv"
                # Create DataFrame and write to CSV

            if(len(df_result) > 0):
                df = pd.DataFrame(df_result)
                
                # Ensure output directory exists
                Path(dbname).parent.mkdir(parents=True, exist_ok=True)
                
                # Write to CSV with proper formatting
                df.to_csv(
                    dbname,
                    index=False,
                    encoding='utf-8',
                    quoting=1, 
                    escapechar='\\'
                )                    
        return result

In [116]:
parser_html = BibleHubHebrewHTMLParser()
html_sources = parser_html.parse_html_folder_biblehub('_scrap_hbrw', None, True)

hosea 3 ( 0.0% )
hosea 13 ( 0.2% )
hosea 2 ( 0.4% )
hosea 7 ( 0.5% )
hosea 9 ( 0.7% )
hosea 6 ( 0.9% )
hosea 5 ( 1.1% )
hosea 10 ( 1.3% )
hosea 1 ( 1.5% )
hosea 12 ( 1.6% )
hosea 4 ( 1.8% )
hosea 11 ( 2.0% )
hosea 14 ( 2.2% )
hosea 8 ( 2.4% )
amos 3 ( 2.6% )
amos 2 ( 2.8% )
amos 7 ( 3.1% )
amos 9 ( 3.4% )
amos 6 ( 3.7% )
amos 5 ( 4.0% )
amos 1 ( 4.3% )
amos 4 ( 4.6% )
amos 8 ( 4.8% )
haggai 2 ( 5.1% )
haggai 1 ( 6.4% )
numbers 3 ( 7.7% )
numbers 33 ( 7.8% )
numbers 13 ( 7.8% )
numbers 28 ( 7.9% )
numbers 34 ( 8.0% )
numbers 21 ( 8.0% )
numbers 2 ( 8.1% )
numbers 7 ( 8.2% )
numbers 27 ( 8.3% )
numbers 9 ( 8.3% )
numbers 6 ( 8.4% )
numbers 30 ( 8.5% )
numbers 25 ( 8.5% )
numbers 23 ( 8.6% )
numbers 18 ( 8.7% )
numbers 32 ( 8.8% )
numbers 26 ( 8.8% )
numbers 29 ( 8.9% )
numbers 36 ( 9.0% )
numbers 5 ( 9.0% )
numbers 17 ( 9.1% )
numbers 10 ( 9.2% )
numbers 1 ( 9.3% )
numbers 12 ( 9.3% )
numbers 35 ( 9.4% )
numbers 4 ( 9.5% )
numbers 20 ( 9.5% )
numbers 11 ( 9.6% )
numbers 22 ( 9.7% )
numbe

In [ ]:
html_sources